Cấu hình kết nối (Configuration)


In [ ]:
import os
# --- CELL 1: CẤU HÌNH KẾT NỐI  ---

# 1. Điền thông tin Event Hubs
EH_NAMESPACE = "nhatlong"      
EH_NAME      = "server-telemetry"      # Tên Event Hub con (đã chốt từ đầu)
SAS_KEY_NAME = "RootManageSharedAccessKey" 

# Key của Event Hubs (Lấy từ chuỗi bạn gửi)
SAS_KEY      = os.environ["EVENTHUB_SAS_KEY"] 

# 2. Điền thông tin Storage Account (Lấy từ code bạn gửi)
STORAGE_ACCOUNT_NAME = "nhatlong"      
STORAGE_ACCOUNT_KEY  = os.environ["AZURE_STORAGE_KEY"]           

# --- TỰ ĐỘNG LẮP GHÉP CHUỖI KẾT NỐI CHUẨN (KHÔNG CẦN SỬA ĐOẠN NÀY) ---
# Code này sẽ tự xóa khoảng trắng thừa (.strip) và thêm EntityPath vào cuối
conn_str = f"Endpoint=sb://{EH_NAMESPACE.strip()}.servicebus.windows.net/;" + \
           f"SharedAccessKeyName={SAS_KEY_NAME.strip()};" + \
           f"SharedAccessKey={SAS_KEY.strip()};" + \
           f"EntityPath={EH_NAME.strip()}"

print("✅ Đã tạo Connection String thành công!")
# In ra chuỗi đã che mật khẩu để kiểm tra
print("Chuỗi kết nối: " + conn_str.replace(SAS_KEY, "******"))

# Cấu hình Spark cho Storage
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)
print("✅ Đã nạp Key Storage thành công!")

Đọc dữ liệu Batch (Inventory) bằng Auto Loader


In [ ]:
# --- XÓA CHECKPOINT CŨ ĐỂ CHẠY LẠI TỪ ĐẦU ---
print("Đang dọn dẹp checkpoint cũ...")
try:
    # Xóa folder checkpoint của Inventory
    dbutils.fs.rm(f"abfss://checkpoint@nhatlong.dfs.core.windows.net/inventory_load", True)
    # Xóa folder checkpoint của Error
    dbutils.fs.rm(f"abfss://checkpoint@nhatlong.dfs.core.windows.net/error_load", True)
    print("✅ Đã xóa sạch checkpoint. Hệ thống sẵn sàng chạy lại như mới!")
except Exception as e:
    print("Không tìm thấy checkpoint cũ hoặc đã xóa rồi. Tiếp tục thôi!")

Nạp Inventory (Sử dụng pathGlobFilter)

In [ ]:
# --- NẠP DỮ LIỆU INVENTORY (BATCH) ---
# Đường dẫn thư mục gốc (Container Bronze)
base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
checkpoint_path_inv = f"abfss://checkpoint@{storage_account_name}.dfs.core.windows.net/inventory_load"
bronze_inventory_table = "bronze_server_inventory"

print(f"Đang đọc file server_inventory.csv từ {base_path}...")

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .option("cloudFiles.schemaLocation", f"{checkpoint_path_inv}/schema")
    # KỸ THUẬT MỚI: Chỉ đọc folder gốc và lọc lấy đúng file csv mình cần
    .option("pathGlobFilter", "server_inventory.csv") 
    .load(base_path)
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path_inv)
    .trigger(availableNow=True)
    .table(bronze_inventory_table)
)

print("✅ Đã nạp xong bảng Inventory vào Bronze!")

Nạp Error Definitions

In [ ]:
# --- NẠP DỮ LIỆU ERROR DEFINITIONS (COMPLEX JSON) ---
checkpoint_path_err = f"abfss://checkpoint@{storage_account_name}.dfs.core.windows.net/error_load"
bronze_error_table = "bronze_error_definitions"

print(f"Đang đọc file error_definitions.json từ {base_path}...")

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path_err}/schema")
    # Lọc lấy đúng file json
    .option("pathGlobFilter", "error_definitions.json")
    .load(base_path)
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path_err)
    .trigger(availableNow=True)
    .table(bronze_error_table)
)

print("✅ Đã nạp xong bảng Error Definitions vào Bronze!")

In [ ]:
%sql
SELECT * FROM bronze_server_inventory;

Đọc dữ liệu Phức tạp (Error Definitions)

In [ ]:
# --- CELL 3: NẠP DỮ LIỆU PHỨC TẠP (ERROR DEFINITIONS) ---

# 1. Cấu hình đường dẫn
base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
checkpoint_path_err = f"abfss://checkpoint@{storage_account_name}.dfs.core.windows.net/error_load"
bronze_error_table = "bronze_error_definitions"

print(f"Đang đọc file error_definitions.json từ {base_path}...")

# 2. Đọc và Ghi vào Bronze
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")        # Quan trọng: Vì file JSON của chúng ta xuống dòng đẹp
    .option("inferSchema", "true")      # Tự động đoán cấu trúc lồng nhau
    .option("cloudFiles.schemaLocation", f"{checkpoint_path_err}/schema") # Nơi lưu cấu trúc
    .option("pathGlobFilter", "error_definitions.json") # Chỉ lấy đúng file này
    .load(base_path)
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path_err)
    .trigger(availableNow=True)         # Chạy xong 1 lần rồi dừng (Batch mode)
    .table(bronze_error_table)
)

print("✅ Đã nạp xong bảng Error Definitions vào Bronze!")

Đọc dữ liệu Streaming (Telemetry) từ Event Hubs


In [ ]:
# --- CELL 4: NẠP DỮ LIỆU STREAMING ---

# 1. Cấu hình Streaming
ehConf = {
  'eventhubs.connectionString': sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(conn_str),
  'eventhubs.consumerGroup': "$Default"
}
# LƯU Ý: Nếu dòng sc._jvm...encrypt vẫn lỗi, hãy xóa nó đi và dùng:
# 'eventhubs.connectionString': conn_str

checkpoint_path_stream = f"abfss://checkpoint@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/telemetry_stream"
bronze_telemetry_table = "bronze_server_telemetry"

print("Đang kết nối tới Event Hubs...")

# 2. Đọc luồng dữ liệu
df_stream = (spark.readStream
  .format("eventhubs")
  .options(**ehConf)
  .load())

# 3. Xử lý & Ghi
from pyspark.sql.functions import col
df_raw = df_stream.select(
    col("body").cast("string").alias("raw_json"), 
    col("enqueuedTime").alias("event_time"),      
    col("offset"),                                
    col("sequenceNumber")
)

query = (df_raw.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", checkpoint_path_stream)
  .table(bronze_telemetry_table)
)

print(f"🚀 Luồng Streaming đang chạy vào bảng '{bronze_telemetry_table}'...")

In [ ]:
%sql
-- 1. Kiểm tra bảng Error Definitions (Phải có dữ liệu JSON lồng nhau)
SELECT * FROM bronze_error_definitions LIMIT 5;

-- 2. Kiểm tra bảng Telemetry (Streaming)
-- Bấm nút Run liên tục vài lần, bạn sẽ thấy số lượng bản ghi (Count) tăng lên
SELECT count(*) as So_luong_ban_ghi FROM bronze_server_telemetry;

-- 3. Xem nội dung dữ liệu Streaming mới nhất
SELECT * FROM bronze_server_telemetry ORDER BY event_time DESC LIMIT 10;